In [4]:
# Get descriptive statistics.

import zipfile
from PIL import Image
import io
import numpy as np

zip_path = "LDIC.zip"

res_real = []
res_fake = []

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    # Only include image files
    files = [
        f for f in z.namelist()
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print(f"Found {len(files)} image files inside ZIP")

    for fname in files:
        count += 1
        if count % 5000 == 0:
            print(f"Processed {count} images...")

        # Extract top-level folder name
        parts = fname.split("/")
        if len(parts) < 2:
            continue  # skip weird entries

        top_folder = parts[0].lower()

        # Determine class
        if top_folder == "real":
            label = "real"
        else:
            label = "fake"

        # Load image
        try:
            with z.open(fname) as f:
                img = Image.open(io.BytesIO(f.read()))
                w, h = img.size
        except:
            print("Warning: could not open", fname)
            continue

        # Store resolution
        if label == "real":
            res_real.append((w, h))
        else:
            res_fake.append((w, h))

print("Done!\n")

# Convert to arrays
real_arr = np.array(res_real)
fake_arr = np.array(res_fake)

print("=== REAL ===")
print("Count:", len(real_arr))
if len(real_arr) > 0:
    print("Min:", real_arr.min(axis=0))
    print("Max:", real_arr.max(axis=0))
    print("Mean:", real_arr.mean(axis=0))

print("\n=== FAKE ===")
print("Count:", len(fake_arr))
if len(fake_arr) > 0:
    print("Min:", fake_arr.min(axis=0))
    print("Max:", fake_arr.max(axis=0))
    print("Mean:", fake_arr.mean(axis=0))

Found 18124 image files inside ZIP
Processed 5000 images...
Processed 10000 images...
Processed 15000 images...
Done!

=== REAL ===
Count: 5890
Min: [256 256]
Max: [256 256]
Mean: [256. 256.]

=== FAKE ===
Count: 12234
Min: [256 256]
Max: [256 256]
Mean: [256. 256.]


In [5]:
# Count png images in each folder

import zipfile
from collections import defaultdict

zip_path = "LDIC.zip"

png_counts = defaultdict(int)

with zipfile.ZipFile(zip_path, 'r') as z:
    for fname in z.namelist():
        if not fname.lower().endswith(".png"):
            continue

        # Example fname: "DALL-E/DALL-E_0001.png"
        parts = fname.split("/")

        if len(parts) < 2:
            continue  # skip weird entries

        top_folder = parts[0]  # e.g., "DALL-E", "Real", "Midjourney"
        png_counts[top_folder] += 1

# Print results
for folder, count in png_counts.items():
    print(f"{folder}: {count} PNG files")

print("\nTotal PNGs:", sum(png_counts.values()))

DeepFaceLab: 1562 PNG files
Midjourney: 930 PNG files
StyleGAN: 1000 PNG files

Total PNGs: 3492


In [6]:
# Unify file types and image resolutions.

import zipfile
from PIL import Image
import io
import os

zip_path = "LDIC.zip"
output_dir = "dataset_4_cleaned"
target_size = (256, 256)

os.makedirs(output_dir, exist_ok=True)

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    # Include only image files
    files = [
        f for f in z.namelist()
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    print(f"Found {len(files)} image files inside ZIP")

    for fname in files:
        count += 1
        if count % 1000 == 0:
            print(f"Processed {count} images...")

        # Extract top-level folder name
        parts = fname.split("/")
        if len(parts) < 2:
            continue  # skip weird entries

        top_folder = parts[0].lower()

        # Determine class
        cls = "real" if top_folder == "real" else "fake"

        class_dir = os.path.join(output_dir, cls)
        os.makedirs(class_dir, exist_ok=True)

        # Load image
        try:
            with z.open(fname) as f:
                img = Image.open(io.BytesIO(f.read())).convert("RGB")
        except:
            print("Warning: could not open", fname)
            continue

        # Resize
        img = img.resize(target_size, Image.LANCZOS)

        # Build a unique base name from the full path inside the ZIP
        rel_path = fname

        # Remove extension
        base_no_ext = os.path.splitext(rel_path)[0]

        # Replace slashes with underscores to make a safe filename
        safe_base = base_no_ext.replace("/", "_").replace("\\", "_")

        suffix = "_real" if cls == "real" else "_fake"
        out_name = safe_base + suffix + ".jpg"
        out_path = os.path.join(class_dir, out_name)

        # Save as uniform JPG
        img.save(out_path, "JPEG", quality=95, subsampling=2)

print("Done! All images converted and resized.")

Found 18124 image files inside ZIP
Processed 1000 images...
Processed 2000 images...
Processed 3000 images...
Processed 4000 images...
Processed 5000 images...
Processed 6000 images...
Processed 7000 images...
Processed 8000 images...
Processed 9000 images...
Processed 10000 images...
Processed 11000 images...
Processed 12000 images...
Processed 13000 images...
Processed 14000 images...
Processed 15000 images...
Processed 16000 images...
Processed 17000 images...
Processed 18000 images...
Done! All images converted and resized.


In [7]:
# Validate updated dataset.

import os
from PIL import Image
import numpy as np

cleaned_path = "dataset_4_cleaned"

real_dir = os.path.join(cleaned_path, "real")
fake_dir = os.path.join(cleaned_path, "fake")

res_real = []
res_fake = []

# --- REAL IMAGES ---
real_files = [f for f in os.listdir(real_dir) if f.lower().endswith(".jpg")]

for i, fname in enumerate(real_files):
    if i % 10000 == 0:
        print(f"Processed {i} real images...")

    img_path = os.path.join(real_dir, fname)
    img = Image.open(img_path)
    w, h = img.size
    res_real.append((w, h))

# --- FAKE IMAGES ---
fake_files = [f for f in os.listdir(fake_dir) if f.lower().endswith(".jpg")]

for i, fname in enumerate(fake_files):
    if i % 10000 == 0:
        print(f"Processed {i} fake images...")

    img_path = os.path.join(fake_dir, fname)
    img = Image.open(img_path)
    w, h = img.size
    res_fake.append((w, h))

print("Done!")

# Convert to arrays for stats
real_arr = np.array(res_real)
fake_arr = np.array(res_fake)

print("=== REAL (cleaned JPG) ===")
print("Count:", len(real_arr))
print("Min:", real_arr.min(axis=0))
print("Max:", real_arr.max(axis=0))
print("Mean:", real_arr.mean(axis=0))

print("\n=== FAKE (cleaned JPG) ===")
print("Count:", len(fake_arr))
print("Min:", fake_arr.min(axis=0))
print("Max:", fake_arr.max(axis=0))
print("Mean:", fake_arr.mean(axis=0))

Processed 0 real images...
Processed 0 fake images...
Processed 10000 fake images...
Done!
=== REAL (cleaned JPG) ===
Count: 5890
Min: [256 256]
Max: [256 256]
Mean: [256. 256.]

=== FAKE (cleaned JPG) ===
Count: 12234
Min: [256 256]
Max: [256 256]
Mean: [256. 256.]


In [9]:
# Color comparison between real and fake images.

import os
import numpy as np
from PIL import Image

cleaned_path = "dataset_4_cleaned"
real_dir = os.path.join(cleaned_path, "real")
fake_dir = os.path.join(cleaned_path, "fake")

def compute_color_stats(folder):
    means = []
    stds = []

    files = [f for f in os.listdir(folder) if f.lower().endswith(".jpg")]

    for i, fname in enumerate(files):
        if i % 10000 == 0:
            print(f"Processed {i} images in {folder}...")

        img = Image.open(os.path.join(folder, fname)).convert("RGB")
        arr = np.array(img) / 255.0  # normalize to [0,1]

        means.append(arr.mean(axis=(0,1)))
        stds.append(arr.std(axis=(0,1)))

    means = np.array(means)
    stds = np.array(stds)

    return means.mean(axis=0), stds.mean(axis=0)

real_mean, real_std = compute_color_stats(real_dir)
fake_mean, fake_std = compute_color_stats(fake_dir)

print("\n=== REAL ===")
print("Mean RGB:", real_mean)
print("Std RGB:", real_std)

print("\n=== FAKE ===")
print("Mean RGB:", fake_mean)
print("Std RGB:", fake_std)

Processed 0 images in dataset_4_cleaned\real...
Processed 0 images in dataset_4_cleaned\fake...
Processed 10000 images in dataset_4_cleaned\fake...

=== REAL ===
Mean RGB: [0.49329232 0.40220979 0.36042206]
Std RGB: [0.25740668 0.22813609 0.21997048]

=== FAKE ===
Mean RGB: [0.4640765  0.39621835 0.36738954]
Std RGB: [0.23700417 0.21018388 0.20649398]
